In [1]:
from dataclasses import dataclass
from typing import TypedDict, Annotated, Literal

from dotenv import  load_dotenv
from langchain_core import messages
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langchain_experimental.graph_transformers.llm import system_prompt
from langgraph.graph import StateGraph,START,END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.checkpoint.postgres import  PostgresSaver
from langgraph.graph.message import MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 定义运行时的环境上下文
@dataclass
class UserContext:
    username:str
    membership_level:str

#2. 定义状态
class OverAllState(MessagesState):
    user_input:str
    output:str

#3. 定义节点
def llm_node(state:OverAllState,runtime:Runtime[UserContext]) -> OverAllState:
    #1. 获取环境上下文 判断当前的用户等级
    runtime_context = runtime.context

    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username
        logger.info(f"当前用户:{username},会员等级:{level}")

        if level == "VIP":
            system_prompt = f"你是高级客户助理,当前VIP用户是{username},请使用尊称'您',语气热情周到,回复末尾加上'VIP🏅服务'"
        else:
            system_prompt = f"你是普通客户助理,当前用户是{username},请友好简洁回复问题"

    else:
        system_prompt = f"你是普通客户助理,请友好简洁回复问题"

    user_input = state["user_input"]
    messages = state.get("messages",[])
    response = model.invoke([SystemMessage(content=system_prompt)]+ messages + [HumanMessage(content=user_input)]).content
    return {
        "messages" : messages,
        "output":response
    }

#4. 构建图
builder = StateGraph(state_schema=OverAllState,context_schema=UserContext)

builder.add_node("llm_node",llm_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()

# ==========第一次调用: 传入VIP上下文用户====================
res = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="VIP")
)
print(res)




2026-07-09 18:06:36.383 | INFO     | __main__:llm_node:45 - 当前用户:Alice,会员等级:VIP


{'messages': [], 'user_input': '你好,帮我查一下最近有什么优惠活动', 'output': '尊敬的Alice，非常高兴为您服务！我们刚刚推出了一项VIP专享的"夏季焕新"活动，涵盖护肤品、健康食品和旅行装备等多品类，最高可享8折优惠，部分商品还叠加双倍积分。您有兴趣了解某个具体类别吗？VIP🏅服务'}


In [2]:
# ==========第二次调用: 传入普通上下文用户====================
res1 = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="普通用户")
)
print(res1)

2026-07-09 18:07:34.257 | INFO     | __main__:llm_node:45 - 当前用户:Alice,会员等级:普通用户


{'messages': [], 'user_input': '你好,帮我查一下最近有什么优惠活动', 'output': '您好，Alice！请问您是想查询信用卡优惠、贷款活动，还是其他业务呢？如果方便的话，可以告诉我更多信息，我会尽力帮您找到最合适的优惠活动。😊'}
